### vLLM for efficient serving

In [1]:
from vllm import LLM, SamplingParams
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import torch
import gc

sys.path.append('./utils')

INFO 04-12 13:40:10 [__init__.py:239] Automatically detected platform cuda.


### Define

In [2]:
model_path="./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
global model_path

In [8]:
from vllm_model_server_start import vllm_model_server_start
from start_siglip_server import start_siglip_server
from siglip_class import SVGMetricEvaluator

### Start vLLM model server

In [4]:
vllm_process= vllm_model_server_start(model_path)

vLLM server started with PID: 10356
Waiting for vLLM model to load...
INFO 04-12 11:57:57 [__init__.py:239] Automatically detected platform cuda.
INFO 04-12 11:57:58 [api_server.py:981] vLLM API server version 0.8.2
INFO 04-12 11:57:58 [api_server.py:982] args: Namespace(subparser='serve', model_tag='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', config='', host='127.0.0.1', port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key='my-api-key', lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, enable_ssl_refresh=False, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plug

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.44s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.47s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.62s/it]



INFO 04-12 11:58:13 [loader.py:447] Loading weights took 3.26 seconds
INFO 04-12 11:58:13 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 3.441487 seconds
INFO 04-12 11:58:19 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/18d80ded3c/rank_0_0 for vLLM's torch.compile
INFO 04-12 11:58:19 [backends.py:425] Dynamo bytecode transform time: 6.17 s


[rank0]:W0412 11:58:20.569000 10438 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 04-12 11:58:21 [backends.py:132] Cache the graph of shape None for later use
INFO 04-12 11:58:39 [backends.py:144] Compiling a graph for general shape takes 19.63 s
INFO 04-12 11:58:48 [monitor.py:33] torch.compile takes 25.80 s in total
INFO 04-12 11:58:48 [kv_cache_utils.py:566] GPU KV cache size: 25,360 tokens
INFO 04-12 11:58:48 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 24.77x
INFO 04-12 11:59:04 [gpu_model_runner.py:1534] Graph capturing finished in 16 secs, took 0.42 GiB
INFO 04-12 11:59:04 [core.py:151] init engine (profile, create kv cache, warmup model) took 51.28 seconds
WARNING 04-12 11:59:04 [config.py:1028] Default sampling parameters have been overridden by the model's Hugging Face generation config recommended from the model creator. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
INFO 04-12 11:59:04 [serving_chat.py:115] Using default chat sampling params from model: {'temperature': 0.6, 'top_p

INFO:     Started server process [10356]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


Checked after 10s...
INFO:     127.0.0.1:52774 - "GET /health HTTP/1.1" 200 OK
vLLM server is ready.
INFO 04-12 11:59:14 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 04-12 11:59:24 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


### OpenAI style client

In [7]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

def get_reponse(description):
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model=model_path,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
        ]
    )
    return chat_response.choices[0].message.content


INFO 04-12 12:00:34 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


### Concurrent calls to vLLM server 

In [8]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=8):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()

#load csv
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df['response'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=12)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


  0%|                                                    | 0/76 [00:00<?, ?it/s]

INFO 04-12 12:00:35 [chat_utils.py:379] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 04-12 12:00:35 [logger.py:39] Received request chatcmpl-37bacb9c20104cbfbb158938afccc9f2: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Vibrant autumn forest',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty

  1%|▌                                           | 1/76 [00:02<03:35,  2.87s/it]

INFO:     127.0.0.1:47964 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 04-12 12:00:37 [logger.py:39] Received request chatcmpl-60b156d161ca4834ac832788f541e4e4: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Snow-capped mountains',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.6, top_p=0.9, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids

  3%|█▏                                          | 2/76 [00:03<02:02,  1.66s/it]

...

 99%|██████████████████████████████████████████▍| 75/76 [00:36<00:00,  1.52it/s]

INFO:     127.0.0.1:47964 - "POST /v1/chat/completions HTTP/1.1" 200 OK


100%|███████████████████████████████████████████| 76/76 [00:37<00:00,  2.02it/s]

INFO:     127.0.0.1:47976 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Total time taken: 37.59 seconds


INFO 04-12 12:01:14 [loggers.py:80] Avg prompt throughput: 30.6 tokens/s, Avg generation throughput: 372.1 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 12:01:24 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 12:01:34 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 12:01:44 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 12:01:54 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cach

### Terminate vLLM Server

In [19]:
#stop vllm server
vllm_process.terminate()
torch.cuda.empty_cache()
gc.collect()
time.sleep(10)

13

In [20]:
from pynvml import nvmlInit, nvmlShutdown, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo

is_running = vllm_process.poll() is None  # True if still running

nvmlInit()
mem = nvmlDeviceGetMemoryInfo(nvmlDeviceGetHandleByIndex(0))
nvmlShutdown()

print(f"vLLM running: {is_running}, GPU used: {mem.used / 1e6:.2f} MB")

vLLM running: False, GPU used: 736.76 MB


In [3]:
server_process=start_siglip_server()

server starting...
Server running: True


In [5]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df.head(1)

,description,gpt_svg,gpt_score_sl
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117


In [8]:
import pandas as pd
import httpx

API_URL = "http://127.0.0.1:8000/evaluate_svg"
API_KEY = "my-api-key"

def send_request(client, prompt, svg):
    try:
        response = client.post(
            API_URL,
            headers={"x-api-key": API_KEY},
            json={"prompt": prompt, "svg": svg},
            timeout=30.0
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def evaluate_all(df):
    with httpx.Client() as client:
        results = []
        for _, row in df.iterrows():
            result = send_request(client, row["description"], row["gpt_svg"])
            results.append(result)
        return results


# Run evaluation
results = evaluate_all(df)
# Append score or error
df["score"] = [r.get("score") if "score" in r else r.get("error") for r in results]


In [9]:
df['score'] = df.apply(lambda row: SVGMetricEvaluator().svg_metric(row['description'], row['gpt_svg']), axis=1)


Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

KeyboardInterrupt: 

In [15]:
df['score'].mean()

0.21148056285059474

In [16]:
df

,description,gpt_svg,gpt_score_sl,score
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,0.013473
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,0.022857
2,"'Warm desert sunset',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.940770,0.000655
3,"'Tropical beach scene',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.884597,0.030664
4,"'Futuristic cityscape',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.986499,0.057330
...,...,...,...,...
71,"'Abstract shapes in blue and green',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.853879,0.019838
72,"'Cloudy sky over rolling hills',","<svg viewBox=""0 0 200 100"" width=""200"" height=...",0.998741,0.063230
73,"'Seaside cliff with crashing waves',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.702254,0.008079
74,"'Cozy fireplace in winter cabin',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.996364,0.641042
